# SAP AI: Colab training

Select **Runtime → Change runtime type → T4 GPU**. Add `DATABASE_URL` under **Secrets** (key icon) and grant this notebook access. If the configured board export is absent, the notebook creates it from the database. It then reads a runtime-local copy and writes checkpoints and run artifacts to Drive.

Running all cells performs the smoke run by default. Full training is opt-in because it can take many hours and consume substantial Drive space.

In [ ]:
REPO_URL = "https://github.com/lgtyqz/sapai-python.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
DRIVE_RUN_DIR = "/content/drive/MyDrive/sapai-runs/policy-improvement-v4-001"  # @param {type:"string"}
BOARDS_JSONL = "/content/drive/MyDrive/sapai-data/boards.jsonl"  # @param {type:"string"}
BOARD_EXPORT_LIMIT = 10000  # @param {type:"integer"}
PACK = "Turtle"  # @param ["Turtle"]
SEED = 2026  # @param {type:"integer"}
REQUIRE_GPU = True  # @param {type:"boolean"}
RUN_FULL_TRAINING = False  # @param {type:"boolean"}
RUN_HUMAN_BENCHMARK = False  # @param {type:"boolean"}
HUMAN_BENCHMARK_DIR = "/content/drive/MyDrive/sapai-runs/human-benchmark"  # @param {type:"string"}
HUMAN_PARTICIPANT_ALIAS = "anonymous"  # @param {type:"string"}
HUMAN_SEED = 2026  # @param {type:"integer"}
assert REPO_URL, "Set REPO_URL to the public Git repository containing this project."
assert BRANCH, "BRANCH cannot be empty."
assert DRIVE_RUN_DIR, "DRIVE_RUN_DIR cannot be empty."
assert BOARDS_JSONL, "BOARDS_JSONL cannot be empty."
assert HUMAN_BENCHMARK_DIR, "HUMAN_BENCHMARK_DIR cannot be empty."
assert HUMAN_PARTICIPANT_ALIAS.strip(), "HUMAN_PARTICIPANT_ALIAS cannot be empty."
assert 2 <= BOARD_EXPORT_LIMIT <= 10000, "BOARD_EXPORT_LIMIT must be between 2 and 10,000."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

repo = Path('/content/sapai-python')
if repo.exists() and not (repo / '.git').is_dir():
    raise RuntimeError(f'{repo} exists but is not a Git checkout; remove or rename it.')
if not repo.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo)],
        check=True,
    )
else:
    origin = subprocess.run(
        ['git', '-C', str(repo), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip().removesuffix('/')
    if origin.removesuffix('.git') != REPO_URL.strip().removesuffix('/').removesuffix('.git'):
        raise RuntimeError(
            f'{repo} was cloned from {origin}, not {REPO_URL}. Restart the runtime or use a new REPO_URL.'
        )
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
os.chdir(repo)
assert sys.version_info >= (3, 11), f'Python 3.11+ is required, found {sys.version}'
GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
print('Repository:', repo)
print('Commit:', GIT_COMMIT)

In [ ]:
%pip install -q -e '.[ml,neon,dev]'
%pip check

# Editable installs add a .pth file that a running Colab kernel may not reload.
# Add the src layout explicitly so this cell works without a runtime restart.
src_dir = str(repo / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
for module_name in tuple(sys.modules):
    if module_name == 'sapai' or module_name.startswith('sapai.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import sapai
print('sapai:', sapai.__file__)
assert Path(sapai.__file__).resolve().is_relative_to(repo.resolve()), (
    f'Imported sapai from the wrong location: {sapai.__file__}'
)

In [ ]:
import tensorflow as tf
from google.colab import userdata
from sapai.data.replay import board_is_pack_compatible
from sapai.data.serialization import read_boards
from sapai.sim.battle import BattleSimulator
from sapai.sim.catalog import Catalog

gpus = tf.config.list_physical_devices('GPU')
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
if REQUIRE_GPU and not gpus:
    raise RuntimeError('No GPU is visible. Select Runtime → Change runtime type → T4 GPU, then restart.')

source_boards = Path(BOARDS_JSONL).expanduser()
needs_board_export = not source_boards.is_file() or source_boards.stat().st_size == 0
if needs_board_export:
    if source_boards.is_file():
        print(f'Replacing empty board export: {source_boards}')
    try:
        database_url = userdata.get('DATABASE_URL')
    except Exception as error:
        raise RuntimeError(
            'Could not read the DATABASE_URL Colab secret. Add it under Secrets and grant notebook access.'
        ) from error
    if not database_url:
        raise RuntimeError('The DATABASE_URL Colab secret is empty.')
    source_boards.parent.mkdir(parents=True, exist_ok=True)
    partial_boards = source_boards.with_name(source_boards.name + '.partial')
    export_environment = os.environ.copy()
    export_environment['DATABASE_URL'] = database_url
    export_result = subprocess.run([
        sys.executable, '-m', 'sapai.cli', 'export-boards',
        '--pack', PACK, '--limit', str(BOARD_EXPORT_LIMIT),
        '--output', str(partial_boards),
    ], check=False, env=export_environment, capture_output=True, text=True)
    if export_result.returncode != 0:
        diagnostic = '\n'.join(
            part.strip() for part in (export_result.stdout, export_result.stderr) if part.strip()
        ).replace(database_url, '[REDACTED DATABASE_URL]')
        raise RuntimeError(
            f'Board export failed with exit code {export_result.returncode}.\n{diagnostic}'
        )
    partial_boards.replace(source_boards)
    del database_url, export_environment
    print(f'Created stable board export: {source_boards}')
if not source_boards.is_file() or source_boards.stat().st_size == 0:
    raise ValueError(f'Board export did not produce data: {source_boards}')
local_boards = Path('/content/sapai-data/boards.jsonl')
local_boards.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_boards, local_boards)
BOARDS_FOR_RUN = str(local_boards)

catalog = Catalog.from_json_dir(repo / 'assets' / 'data')
pack_labeled_boards = [board for board in read_boards(local_boards) if board.pack == PACK]
boards = [
    board for board in pack_labeled_boards
    if board_is_pack_compatible(board, catalog, PACK)
]
excluded_boards = len(pack_labeled_boards) - len(boards)
if excluded_boards:
    print(
        f'WARNING: excluded {excluded_boards:,} {PACK}-labeled boards containing known cross-pack pets.'
    )
if not boards:
    raise ValueError(f'{source_boards} contains no compatible {PACK!r} boards.')
simulator = BattleSimulator(catalog)
vanilla_fallbacks = {}
perk_fallbacks = {}
for board in boards:
    for pet in board.team.slots:
        if pet is not None and pet.metadata.get('vanilla_fallback'):
            key = (pet.id, pet.name)
            vanilla_fallbacks[key] = vanilla_fallbacks.get(key, 0) + 1
    try:
        simulator.assert_team_supported(board.team)
    except Exception as error:
        pets = [(pet.id, pet.name) for pet in board.team.slots if pet is not None]
        raise RuntimeError(
            f'Unsupported board from {local_boards}: replay_id={board.replay_id!r}, '
            f'side={board.side!r}, turn={board.turn}, pets={pets}'
        ) from error
    for pet in board.team.slots:
        if pet is not None and pet.metadata.get('perk_fallback'):
            key = pet.perk or '<missing perk>'
            perk_fallbacks[key] = perk_fallbacks.get(key, 0) + 1
if vanilla_fallbacks:
    print('WARNING: unknown pet IDs using recorded stats without abilities:')
    for pet, count in sorted(vanilla_fallbacks.items()):
        print(f'  {pet}: {count:,} board slots')
if perk_fallbacks:
    print('WARNING: perks using no-effect fallback:')
    for perk, count in sorted(perk_fallbacks.items()):
        print(f'  {perk}: {count:,} board slots')
print(f'Validated {len(boards):,} {PACK} boards; local snapshot: {local_boards}')
del boards
Path(DRIVE_RUN_DIR).parent.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(Path(DRIVE_RUN_DIR).parent).free / 2**30
print(f'Drive free space: {free_gib:.1f} GiB')
if free_gib < 2:
    raise RuntimeError('Less than 2 GiB is free in Drive; choose another run directory or free space.')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## Small end-to-end smoke run

This validates policy checkpointing, complete Arena rollouts, simulator-scored search leaves, and search distillation before a long run.

In [ ]:
smoke_dir = str(
    Path(DRIVE_RUN_DIR).with_name(Path(DRIVE_RUN_DIR).name + f'-smoke-{GIT_COMMIT[:8]}')
)
smoke = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', smoke_dir, '--pack', PACK,
    '--bootstrap-episodes', '2', '--bootstrap-exploration', '0.10',
    '--bootstrap-epochs', '1', '--search-episodes', '1',
    '--search-epochs', '1', '--search-iterations', '1', '--search-simulations', '4',
    '--bootstrap-replay-fraction', '0.35',
    '--search-candidates', '8', '--battle-evaluation-simulations', '2',
    '--validation-episodes', '1', '--test-episodes', '1',
    '--batch-size', '32', '--seed', str(SEED),
]
subprocess.run(smoke, check=True)

## Full training sequence

Edit counts here for the dataset and time budget. Rerunning with the same configuration and `DRIVE_RUN_DIR` reuses completed datasets and rollout episodes, then restores model and optimizer state from the latest Drive checkpoint. An interrupted epoch restarts deterministically from its beginning. The v4 target schema groups source/target variants during policy choice, permits intentional early End Turn decisions, and rejects roll-collapse checkpoints, so it requires a fresh `DRIVE_RUN_DIR`; older checkpoints are rejected explicitly.

After a runtime disconnect, rerun the configuration, Drive, repository, installation, and validation cells (cells 1–5), skip the smoke cell, then rerun this full-training cell with `RUN_FULL_TRAINING=True`. The cell prints elapsed-time progress for checkpointed Arena episodes, simulator-scored search rollouts, and every saved policy-training epoch.

In [ ]:
for model_name in ('policy-model',):
    checkpoint_dir = str(Path(DRIVE_RUN_DIR) / model_name / 'checkpoints')
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        completed = int(tf.train.load_variable(
            latest_checkpoint, 'completed_epochs/.ATTRIBUTES/VARIABLE_VALUE'
        ))
        print(f'{model_name}: restoring completed epoch {completed} from {latest_checkpoint}')
    else:
        print(f'{model_name}: no checkpoint found yet')

full = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', DRIVE_RUN_DIR, '--pack', PACK,
    '--bootstrap-episodes', '1000', '--bootstrap-exploration', '0.10',
    '--bootstrap-epochs', '20', '--search-episodes', '250',
    '--search-epochs', '5', '--search-iterations', '3', '--search-simulations', '32',
    '--bootstrap-replay-fraction', '0.35',
    '--search-candidates', '8', '--battle-evaluation-simulations', '16',
    '--validation-episodes', '20', '--test-episodes', '50',
    '--batch-size', '128', '--seed', str(SEED),
    '--progress',
]
if RUN_FULL_TRAINING:
    subprocess.run(full, check=True)
else:
    print('Set RUN_FULL_TRAINING=True when the smoke run succeeds.')

## Visualize the latest policy

The generated HTML contains both shops and battles and is also saved with the run artifacts.

In [ ]:
active_run = DRIVE_RUN_DIR if RUN_FULL_TRAINING else smoke_dir
visualization = str(Path(active_run) / 'arena.html')
subprocess.run([
    sys.executable, '-m', 'sapai.cli', 'visualize-arena',
    '--boards', BOARDS_FOR_RUN, '--pack', PACK, '--policy', 'model',
    '--policy-weights', str(Path(active_run) / 'policy-model'),
    '--seed', str(SEED), '--output', visualization,
], check=True)
archive = Path(active_run) / 'arena-visualization.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ('arena.html', 'sapai.css', 'sapai.js'):
        bundle.write(Path(active_run) / name, name)
    for asset in (Path(active_run) / 'sapai-assets').rglob('*'):
        if asset.is_file():
            bundle.write(asset, asset.relative_to(active_run))
print(f'Visualization: {visualization}')
print(f'Portable archive: {archive}')
print('Open or download it from the Colab Files pane. Inline display is omitted because the HTML uses sibling CSS, JavaScript, and sprite files.')

## Human Arena benchmark

This optional section opens a card-driven Arena game inside Colab and uses the same compatible `BOARDS_FOR_RUN` opponent pool as model rollouts. Every accepted move is checkpointed to `HUMAN_BENCHMARK_DIR`; there is no undo or restart, and a new game becomes available only after the current Arena run ends.

For a participant-only session, set `REQUIRE_GPU=False` and `RUN_HUMAN_BENCHMARK=True`, run cells 1–5, skip the training and model-visualization cells, then run the launcher below. After a runtime disconnect, rerun cells 1–5 and this launcher to restore the exact shop or battle-review state. Reusing an existing directory resumes compatible data; if repository, rules, or board settings changed, the launcher preserves that data and selects a deterministic suffixed directory. Colab callbacks work only for output executed in the current runtime.

In [ ]:
if RUN_HUMAN_BENCHMARK:
    from sapai.training.human import (
        HumanArenaSession,
        HumanBenchmarkConfig,
        sha256_file,
    )
    from sapai.sim.shop import ShopEnvironment
    from sapai.training.population import load_opponent_boards, split_opponent_populations
    from sapai.visualization import display_human_arena

    human_benchmark_dir = Path(HUMAN_BENCHMARK_DIR).expanduser()
    training_run_dir = Path(DRIVE_RUN_DIR).expanduser().resolve()
    human_benchmark_dir = human_benchmark_dir.resolve()
    if (
        human_benchmark_dir == training_run_dir
        or training_run_dir in human_benchmark_dir.parents
        or human_benchmark_dir in training_run_dir.parents
    ):
        raise ValueError('HUMAN_BENCHMARK_DIR must not overlap DRIVE_RUN_DIR.')
    human_populations = split_opponent_populations(
        load_opponent_boards(BOARDS_FOR_RUN, catalog, PACK), seed=SEED,
    )
    human_population = human_populations.test
    human_config = HumanBenchmarkConfig(
        output_dir=human_benchmark_dir,
        participant_alias=HUMAN_PARTICIPANT_ALIAS,
        pack=PACK,
        seed=HUMAN_SEED,
        boards_sha256=sha256_file(BOARDS_FOR_RUN),
        board_count=len(human_population.boards),
        repository_commit=GIT_COMMIT,
    )
    human_session = HumanArenaSession.create_or_resume(
        ShopEnvironment(catalog),
        BattleSimulator(catalog),
        human_population,
        human_config,
        version_on_mismatch=True,
    )
    active_human_benchmark_dir = human_session.config.directory
    if active_human_benchmark_dir != human_benchmark_dir:
        print(f'Existing benchmark preserved; using compatible directory: {active_human_benchmark_dir}')
    HUMAN_CALLBACK = display_human_arena(human_session, repo / 'assets')
    print(f'Human benchmark artifacts: {active_human_benchmark_dir}')
else:
    print('Set RUN_HUMAN_BENCHMARK=True to launch or resume the human Arena benchmark.')